# I2R - Implementacion guiada en Jupyter

Este notebook es una version ordenada para seguir trabajando como antes, celda a celda.

Los scripts que hay en la carpeta no sustituyen al notebook: solo evitan repetir codigo y hacen que los experimentos sean reproducibles. Desde aqui los puedes ejecutar sin salir de Jupyter.


## 1. Comprobar carpeta del proyecto

Abre Jupyter desde la carpeta `I2R`. Esta celda debe mostrar los ficheros `train.tsv`, `valid.tsv` y `test.tsv`.


In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
print(PROJECT_DIR)
print((PROJECT_DIR / 'train.tsv').exists(), (PROJECT_DIR / 'valid.tsv').exists(), (PROJECT_DIR / 'test.tsv').exists())


c:\Users\victor.moreno.villan\Desktop\I2R
True True True


## 2. Cargar datos y ver distribucion

Esta parte equivale a lo que ya tenias, pero usando la funcion comun del proyecto.


In [2]:
import pandas as pd
from src.data_utils import load_liar

train, valid, test = load_liar(PROJECT_DIR, binary=True)
print(train.shape, valid.shape, test.shape)
train[['label', 'label_binary', 'statement']].head()


(10240, 15) (1284, 15) (1267, 15)


,label,label_binary,statement
0,false,0,Says the Annies List political group supports ...
1,half-true,1,When did the decline of coal start? It started...
2,mostly-true,1,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,0,Health care reform legislation is likely to ma...
4,half-true,1,The economic turnaround started at the end of ...


In [3]:
train['label_binary'].value_counts().rename(index={0: 'fake', 1: 'real'})


label_binary
real    5752
fake    4488
Name: count, dtype: int64

## 3. Ejecutar baselines clasicas

Esto entrena TF-IDF + Logistic Regression y TF-IDF + Linear SVM. Genera `results_baselines.csv` y matrices de confusion en `outputs/`.


In [4]:
%run run_baselines.py --project-dir .


                       Model Split  Accuracy  Macro Precision  Macro Recall  Macro F1
TF-IDF + Logistic Regression valid  0.605140         0.604824      0.604965  0.604812
TF-IDF + Logistic Regression  test  0.603788         0.598339      0.598917  0.598530
         TF-IDF + Linear SVM valid  0.575545         0.574742      0.574690  0.574707
         TF-IDF + Linear SVM  test  0.610103         0.604324      0.604723  0.604485

Saved: C:\Users\victor.moreno.villan\Desktop\I2R\results_baselines.csv
Saved: C:\Users\victor.moreno.villan\Desktop\I2R\outputs\baseline_classification_reports.md


In [5]:
baseline_results = pd.read_csv(PROJECT_DIR / 'results_baselines.csv')
baseline_results


,Model,Split,Accuracy,Macro Precision,Macro Recall,Macro F1
0,TF-IDF + Logistic Regression,valid,0.605140,0.604824,0.604965,0.604812
1,TF-IDF + Logistic Regression,test,0.603788,0.598339,0.598917,0.598530
2,TF-IDF + Linear SVM,valid,0.575545,0.574742,0.574690,0.574707
3,TF-IDF + Linear SVM,test,0.610103,0.604324,0.604723,0.604485


## 4. Entrenar Transformer

Empieza con DistilBERT porque es mas ligero. En CPU puede tardar bastante. Si va muy lento, baja `--epochs 1`.


In [6]:
# Ejecuta esta celda cuando quieras lanzar el entrenamiento real.
# En CPU puede tardar bastante.
%run train_transformer.py --project-dir . --model-name distilbert-base-uncased --epochs 3 --batch-size 8 --learning-rate 2e-5


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\victor.moreno.villan\Desktop\I2R\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\victor.moreno.villan\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.459878,0.758102,3,0.640966,0.642201,0.637375,0.636197


c:\Users\victor.moreno.villan\Desktop\I2R\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.459878,0.769807,3,0.647987,0.640965,0.633851,0.634544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

                  Model Split  Accuracy  Macro Precision  Macro Recall  Macro F1
distilbert-base-uncased valid  0.640966         0.642201      0.637375  0.636197
distilbert-base-uncased  test  0.647987         0.640965      0.633851  0.634544
Saved model and tokenizer under: C:\Users\victor.moreno.villan\Desktop\I2R\models\distilbert-base-uncased\best_model
Saved metrics: C:\Users\victor.moreno.villan\Desktop\I2R\outputs\results_transformer.csv


In [7]:
transformer_results_path = PROJECT_DIR / 'outputs' / 'results_transformer.csv'
if transformer_results_path.exists():
    transformer_results = pd.read_csv(transformer_results_path)
    display(transformer_results)
else:
    print('Todavia no existe outputs/results_transformer.csv. Ejecuta primero la celda de entrenamiento.')


,Model,Split,Accuracy,Macro Precision,Macro Recall,Macro F1
0,distilbert-base-uncased,valid,0.640966,0.642201,0.637375,0.636197
1,distilbert-base-uncased,test,0.647987,0.640965,0.633851,0.634544


## 5. Explicabilidad con LIME

Ejecuta esta parte despues de entrenar DistilBERT. Genera HTMLs en `outputs/explanations/`.


In [8]:
%run explain_lime.py --project-dir . --model-dir ./models/distilbert-base-uncased/best_model --num-samples 3


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

 test_index true_label                                                            statement                                                                explanation_file
          0       real Building a wall on the U.S.-Mexico border will take literally years. C:\Users\victor.moreno.villan\Desktop\I2R\outputs\explanations\lime_test_0.html
          1       fake      Wisconsin is on pace to double the number of layoffs this year. C:\Users\victor.moreno.villan\Desktop\I2R\outputs\explanations\lime_test_1.html
          2       fake                  Says John McCain has done nothing to help the vets. C:\Users\victor.moreno.villan\Desktop\I2R\outputs\explanations\lime_test_2.html
Saved explanations under: C:\Users\victor.moreno.villan\Desktop\I2R\outputs\explanations


In [9]:
lime_summary = PROJECT_DIR / 'outputs' / 'explanations' / 'lime_summary.csv'
if lime_summary.exists():
    display(pd.read_csv(lime_summary))
else:
    print('Todavia no hay resumen LIME. Ejecuta primero la celda anterior.')


,test_index,true_label,statement,explanation_file
0,0,real,Building a wall on the U.S.-Mexico border will...,C:\Users\victor.moreno.villan\Desktop\I2R\outp...
1,1,fake,Wisconsin is on pace to double the number of l...,C:\Users\victor.moreno.villan\Desktop\I2R\outp...
2,2,fake,Says John McCain has done nothing to help the ...,C:\Users\victor.moreno.villan\Desktop\I2R\outp...


## 6. Que poner en la memoria

Cuando tengas resultados, la historia experimental queda asi:

1. Dataset LIAR y transformacion binaria.
2. Baselines clasicas: TF-IDF + Logistic Regression / Linear SVM.
3. Transformer: DistilBERT fine-tuned.
4. Comparacion con Accuracy, Macro Precision, Macro Recall y Macro F1.
5. Explicabilidad: ejemplos concretos con LIME.
